# 👗 From Noise to Fashion: Image Generation with Diffusion

Welcome! This notebook walks you through building and training an **conditional image generation model** on FashionMNIST: you can tell it which clothing category to generate, and it will create new images of that type from scratch.

Today you will build the core component of the "Graphic Designer Agent" from the full AI-run magazine application. From thin air, you will be able to generate images of clothes by using a model that learns to reconstruct a corrupted image.

This should feel almost like magic! Keep in mind, though, that due to time and resource constraints we will work on much simpler scenarios. The pipeline you build today is the foundation on which modern state-of-the-art image generation models are built. Those models, such as Stable Diffusion, use the very same approach, but are trained on billions of images across ~256 high-end GPUs for several weeks.

**Pipeline:**

1. 👗 **Load the data**: set up FashionMNIST with 10 clothing classes
2. 💨 **Forward diffusion**: mathematically destroy images by adding Gaussian noise
3. 🔬 **Visualize the noise**: watch a clean image dissolve into static, step by step
4. 🧠 **Train a denoising model**: teach a UNet to predict and reverse the noise
5. 🎨 **Generate new fashion**: sample class-conditional images from pure noise
6. 🎞️ **Watch it denoise**: visualize the full reverse diffusion journey
7. 🌐 **Interactive web app**: use your trained model to generate fashion on demand

By the end, you'll be able to prompt the model for any of the 10 clothing categories, giving you a hands-on look at the core technology behind modern image generation systems.

> ⚠️ **GPU STRONGLY RECOMMENDED**: training will be very slow on CPU!
>
> Recommended runtimes in Colab: **L4** or **T4**
> - *Runtime → Change runtime type → Hardware accelerator → GPU*
> - If **L4 / T4** do not appear as options, contact a **Teaching Assistant**.

---
## 0 📦 Imports

We import PyTorch for building and training the model, torchvision for the FashionMNIST dataset, and matplotlib for visualizations.

In [ ]:
#@title Fetch necessary utilities { display-mode: "form" }
import subprocess

subprocess.run(["curl", "-fsSL", "-O",
                "https://raw.githubusercontent.com/eth-bmai-hs26/w3-project-public/refs/heads/main/notebook/utils.py"], check=True)

In [ ]:
import torch
import torch.nn.functional as F
from torch import optim
import torchvision
from torch.utils.data import DataLoader
from torchvision import transforms
import matplotlib.pyplot as plt
import utils
import subprocess, os

device = "cpu"
if torch.cuda.is_available():           # NVIDIA GPU (Colab L4 / T4)
    device = "cuda"
elif torch.backends.mps.is_available(): # Apple M-series chip
    device = "mps"

if device == "cpu" or device == "mps":
    print("=" * 60)
    print("⚠️  WARNING: No GPU detected!")
    print("=" * 60)
else:
    print(f"✅ Using device: {device}")


---
## 1 👗 Load the Data

Before we can generate fashion images, we need to show the model what fashion looks like. **FashionMNIST** contains 60,000 grayscale training images across 10 clothing categories, each 28×28 pixels.

| Label | 0 | 1 | 2 | 3 | 4 | 5 | 6 | 7 | 8 | 9 |
|-------|---|---|---|---|---|---|---|---|---|---|
| **Class** | T-shirt/top | Trouser | Pullover | Dress | Coat | Sandal | Shirt | Sneaker | Bag | Ankle boot |

We normalize each image to the range **[-1, 1]** (from raw [0, 255] pixel values). This matters for diffusion models: the noise process is defined over a zero-centered Gaussian, so keeping pixel values zero-centered keeps everything consistent.


In [ ]:
IN_CHANNELS = 1
IMG_SIZE = 32
CLASS_NAMES = [
    'T-shirt/top', 'Trouser', 'Pullover', 'Dress', 'Coat',
    'Sandal', 'Shirt', 'Sneaker', 'Bag', 'Ankle boot'
]
NUM_CLASSES = len(CLASS_NAMES)
CLASS_TO_IDX = {name: idx for idx, name in enumerate(CLASS_NAMES)}

transform = transforms.Compose([
    transforms.Resize(IMG_SIZE),
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,))
])
train_dataset = torchvision.datasets.FashionMNIST(
    root="./data", train=True, download=True, transform=transform
)

batch_size = 1024
train_loader = DataLoader(dataset=train_dataset, batch_size=batch_size, shuffle=True)

# Let's look at one image to confirm
image, label = train_dataset[0]
print("Image shape:", image.shape)
print(f"Label: {label} ({CLASS_NAMES[label]})")
print(f"Pixel value range: {image.min()} to {image.max()}")

In [ ]:
utils.show_class_examples(train_dataset, CLASS_NAMES)

---
## 2 💨 Forward Diffusion

The key idea behind diffusion models is elegantly destructive: take a clean image and **gradually add Gaussian noise** over many timesteps until nothing recognizable remains. Then train a neural network to predict that noise.

There is an equation that lets us corrupt an image to any noise level $t$ directly:

$$x_{t} = \sqrt{\bar{\alpha}_{t}} \, x_{0} + \sqrt{1-\bar{\alpha}_{t}} \, \varepsilon, \;\;\; \varepsilon \sim \mathcal{N}(0,I).$$

Here $\bar{\alpha}_t$ controls how much of the original image survives at timestep $t$:
- When $\bar{\alpha}_t \approx 1$ (early timesteps): the image is nearly unchanged
- When $\bar{\alpha}_t \approx 0$ (late timesteps): the image is almost pure Gaussian noise

Note that $\varepsilon$ is the noise pattern we sample, and $x_0$ is a real training image. **Both are scaled before being blended.** The formula is not "image plus noise" but a weighted mix of the two, with weights chosen so that $x_t$ keeps roughly the same variance at every $t$.

> **Why is this useful?** Training samples a random timestep $t$ for every image in the batch. This formula lets us jump directly to any noise level, making training fast.

In [ ]:
# Hyperparameters for the diffusion process
TIMESTEPS = 1_000
beta_start = 0.0001
beta_end = 0.02

# Define the beta schedule (variance)
betas = torch.linspace(beta_start, beta_end, TIMESTEPS, device=device)

# Pre-calculate alphas and other values for the formula
alphas = 1. - betas
alphas_bar = torch.cumprod(alphas, dim=0) # Computes comulative product of alphas

sqrt_alphas_bar = torch.sqrt(alphas_bar)
sqrt_one_minus_alphas_bar = torch.sqrt(1. - alphas_bar)

# Helper function to get the correct values for a batch of timesteps
def get_values_at_timestep(values, t):
    """
    Gathers the values from the `values` tensor at the indices specified by `t`.
    Reshapes the output to be compatible for broadcasting with image tensors.
    """
    batch_size = t.shape[0]
    # Gather values and reshape to (batch_size, 1, 1, 1) for broadcasting
    return values.gather(-1, t).reshape(batch_size, 1, 1, 1)

### 🎯 2.1 Implement the forward noising function

Complete the `add_noise` function by filling in the three missing values.

> 💡 Hints:
> - Use `get_values_at_timestep` to extract precomputed values at timestep `t`, e.g. `a_t = get_values_at_timestep(a, t)`
> - The formula to corrupt an image to noise level $t$ is: $x_t = \sqrt{\bar{\alpha}_t} \, x_0 + \sqrt{1 - \bar{\alpha}_t} \, \varepsilon$
> - The two precomputed tensors you need are `sqrt_alphas_bar` and `sqrt_one_minus_alphas_bar`

In [ ]:
# The Forward Noising Function
def add_noise(x_0, t):
    """
    Applies noise to an image x_0 at a specific timestep t.
    
    Args:
        x_0: The original image tensor
        t: timestep tensor
    
    Returns:
        x_t: The noisy image at timestep t
        noise: The noise that was added to the image
    """
    # Sample random noise from a standard Gaussian
    noise = torch.randn_like(x_0)

    # 🎯 Get the pre-calculated values for the given timesteps 
    sqrt_alphas_bar_t = ...
    sqrt_one_minus_alphas_bar_t = ...

    # 🎯 Apply the formula to get the noisy image
    x_t = ...
    
    # 🎯
    return ..., ...

---
## 3 🔬 Visualize the Noising Process

Let's see the forward process in action. We'll take a single image and plot it at 10 evenly-spaced timesteps from $t=0$ (clean) to $t=T-1$ (pure noise).

**What to look for:**
- **Early timesteps**: the original image is clearly recognizable, just slightly grainier
- **Middle timesteps**: fine details blur; shapes start to melt into the background
- **Late timesteps**: the image is unrecognizable, pure Gaussian noise

This progression is exactly what the model has to *undo* during generation.

### 🎯 3.1 Visualize the noising process

Run the cell below to apply `add_noise` at 10 evenly-spaced timesteps and plot the results.

> 💡 What to check: the image should be nearly clean at $t=0$ and indistinguishable from random noise by $t=999$. If it looks wrong, revisit your `add_noise` implementation.

In [ ]:
# Visualization of the forward pass
image, _ = train_dataset[0] # Get first image
image = image.to(device).unsqueeze(0) # Add batch dimension and move to device

num_steps = 10  # Number of noise levels to visualize
vis_timesteps = torch.linspace(0, TIMESTEPS - 1, num_steps).long().tolist()
noisy_images = []
titles = []

# Generate noisy images at specified timesteps
for timestep in vis_timesteps:
    t = torch.tensor([timestep], device=device) # Convert to tensor and move to device
    
    # 🎯 Use add_noise to corrupt the image
    noisy_img, _ = ...
    
    noisy_images.append(noisy_img)
    titles.append(f"t = {timestep}")

print("Visualizing the forward noising process:")
utils.show_images(torch.cat(noisy_images, dim=0), titles=titles)

---
## 4 🧠 Reverse Diffusion: Training the Noise Predictor

Now for the clever part. We've seen how to destroy an image. Reversing that
destruction takes two pieces:

1. **A fixed formula: the sampler.** It computes $x_{t-1}$ from $x_t$. But in order to
   step back from $x_t$, we need to know the noise pattern that went into $x_t$.

2. **A neural network predicts that noise pattern.** A UNet is trained so given a noisy image $x_t$ and its timestep $t$, it predicts the noise pattern $\varepsilon$ that was sampled to produce it.
   **Input:** an image. **Output:** an image-shaped noise estimate. 

So the division of labour is: the UNet predicts the sampled noise
$\varepsilon_\theta(x_t, t)$, and the sampling equation below uses that
prediction to recover $x_{t-1}$ from $x_t$. Repeat the pair of steps 1,000 times
starting from pure static, and a fashion item emerges!

The model is **class-conditional**: we pass the clothing category label alongside
the noisy image, so its noise prediction is class-aware. That is how you steer
generation toward a specific category at sampling time.

### 4.1 🎯-🏗️ Define the UNet

`SimpleUNet` is a lightweight UNet with skip connections. It takes a noisy image $x_t$, the timestep $t$, and the class label as inputs, and outputs a **predicted noise tensor of the same shape**.

The training objective is simple: minimize the MSE between the predicted noise and the actual noise that was added. This is the core of the DDPM training algorithm.

| Input | What it is |
|-------|------------|
| `x_t` | The noisy image at timestep $t$ |
| `t` | The timestep, which tells the model how noisy the input is |
| `labels` | The class label, which steers denoising toward a specific category |

In [ ]:
import math
import torch.nn as nn

class SimpleUNet(nn.Module):
    """
    A simple UNet with Class Conditioning.

    args:
    - base_ch: number of channels in the first layer (doubles every downsample)
    - emb_dim: dimension of the time and class embedding
    - num_classes: number of classes for conditioning
    - in_channels: number of input image channels (1=grayscale, 3=RGB)

    note: output channels are always the same as input channels, since we're predicting noise which has the same shape as the input image.
    """
    def __init__(self, base_ch=64, emb_dim=64, num_classes=10, in_channels=1):
        super().__init__()
        
        self.base_ch = base_ch
        self.emb_dim = emb_dim
        
        # 🎯 Fill in input and output channels
        self.in_ch = ...
        self.out_ch = ...
        

        # This learns a unique vector of size 'emb_dim' for each class (0-9).
        self.label_emb = nn.Embedding(num_classes, emb_dim)

        # This MLP combines the time and label embeddings into a single conditioning embedding that will be used in the ConvBlocks.
        self.cond_mlp = nn.Sequential(
            nn.Linear(emb_dim, emb_dim * 2),
            nn.SiLU(),
            nn.Linear(emb_dim * 2, emb_dim),
        )

        # ↘️ Encoder --------------------------------------------------------------------------------------
        self.enc1 = ConvBlock(self.in_ch, base_ch, emb_dim)
        self.down1 = nn.Conv2d(base_ch, base_ch, 4, stride=2, padding=1)
        
        # 🎯 Fill enc2 and down2
        # Hint: recall that in enc, the number of output channels is double the number of input channels
        # Hint: in the downsample, you can keep the same number of filter size, stride, and padding as down1
        self.enc2 = ...
        self.down2 = ...
        
        # 🎯 Fill enc3 and down3
        self.enc3 = ...
        self.down3 = ...
        # End of encoder --------------------------------------------------------------------------------

        # 🪢 Bottleneck
        # 🎯 Fill the bottleneck layer
        # Hint: use a ConvBlock, with the same number of input and output channels
        self.bot = ...

        # ↗️ Decoder --------------------------------------------------------------------------------------
        self.up3 = nn.ConvTranspose2d(base_ch * 4, base_ch * 4, 4, stride=2, padding=1)
        self.dec3 = ConvBlock(base_ch * 4 + base_ch * 4, base_ch * 2, emb_dim)
        
        # 🎯 Fill up2 and dec2
        # Hint: in the upsample, you can keep the same number of filter size, stride, and padding as up3
        # Hint: for dec2, don't forget to account for the skip connection
        self.up2 = ...
        self.dec2 = ...
        
        # 🎯 Fill up1 and dec1
        # Hint: in dec 1, don't forget to account for the skip connection
        self.up1 = ...
        self.dec1 = ...
        # End of decoder --------------------------------------------------------------------------------------

        self.out = nn.Conv2d(base_ch, self.out_ch, 1)

    def forward(self, x, t, labels):
        """
        x: (B, C, H, W)
        t: (B,)
        labels: (B,) IntTensor containing class indices (0-9)
        """
        # create time and label embeddings and combine them into a single conditioning embedding
        time_emb = sinusoidal_time_embedding(t, self.emb_dim)
        label_emb = ...     # 🎯 Create the label embedding using self.label_emb
        cond_emb = ... # 🎯 Sum time and label embeddings and pass through self.cond_mlp

        # ↘️ encoder
        e1 = self.enc1(x, cond_emb)
        d1 = self.down1(e1)
        
        # 🎯 Use self.enc2 and self.down2 to go one level deeper
        # Hint: recall that enc2 takes the output of down1 as input
        e2 = ...
        d2 = ...
        
        # 🎯 Use self.enc3 and self.down3 to go one level deeper
        e3 = ...
        d3 = ...

        # 🪢 bottleneck
        b = self.bot(d3, cond_emb)

        # ↗️decoder
        u3 = self.up3(b)
        u3 = torch.cat([u3, e3], dim=1)     # skip connection: concatenate the upsampled features with the corresponding encoder features
        u3 = self.dec3(u3, cond_emb)
        
        # 🎯 Fill up2 and dec2
        # Hint: in up2, input is the output of dec3
        u2 = self.up2(...)
        u2 = torch.cat(..., dim=1)
        u2 = ...
        
        # 🎯 Fill up1 and dec1
        u1 = ...
        u1 = ...
        u1 = ...

        return self.out(u1)

class ConvBlock(nn.Module):
    """
    Two convolutional layers followed by FiLM conditioning.

    FiLM (Feature-wise Linear Modulation) lets the network behave differently
    depending on the timestep and class, by scaling and shifting the feature map:

        output = features * (1 + scale) + shift

    where scale and shift are predicted from the conditioning embedding.
    """
    def __init__(self, in_ch, out_ch, emb_dim):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, 3, padding=1),
            nn.GroupNorm(1, out_ch),
            nn.SiLU(),
            nn.Conv2d(out_ch, out_ch, 3, padding=1),
            nn.GroupNorm(1, out_ch),
            nn.SiLU(),
        )
        # 🎯 predict per-channel scale and shift from the conditioning embedding
        # Hint: use 2 identical linear layers, one for scale and one for shift
        # Hint: input dimension is dim of the conditioning embedding, output dimension is the number of output channels
        self.to_scale = ...
        self.to_shift = ...

    def forward(self, x, cond_emb):
        # First we apply the convolutional layers to get the features
        h = self.conv(x)
        
        # Then we predict the scale and shift from the conditioning embedding
        scale = self.to_scale(cond_emb).unsqueeze(-1).unsqueeze(-1)
        shift = self.to_shift(cond_emb).unsqueeze(-1).unsqueeze(-1)
        
        # 🎯 Finally we apply a modulation to the features using the predicted scale and shift
        # You can think of this as the network saying "at this timestep and for this class, I want to amplify these features and suppress these other features"
        # Hint: output = features * (1 + scale) + shift
        return ...

# NO EDIT ARE NEEDED BELOW THIS LINE (for this cell)
def sinusoidal_time_embedding(timesteps: torch.Tensor, dim: int) -> torch.Tensor:
    """
    Transformer-style sinusoidal positional encoding for diffusion timesteps.
    This is needed because the model needs to know which timestep it's at in order to predict the noise correctly.

    timesteps: (B,) long
    returns:   (B, dim)
    """
    device = timesteps.device
    half = dim // 2
    emb = math.log(10000) / (half - 1)
    emb = torch.exp(torch.arange(half, device=device) * -emb)
    emb = timesteps.float().unsqueeze(1) * emb.unsqueeze(0)
    emb = torch.cat([torch.sin(emb), torch.cos(emb)], dim=1)
    if dim % 2 == 1:
        emb = F.pad(emb, (0, 1))
    return emb  # (B, dim)

### 4.2 🔍 Shape Check

Before training, we confirm that the output has **exactly the same shape as the input**, because the model predicts one noise value per pixel. If the shapes differ, something in the encoder/decoder is misaligned.

In [ ]:
# Sanity check: output shape must equal input shape.
# We use a tiny base_ch here so this runs instantly and uses almost no memory.
m = SimpleUNet(base_ch=8, emb_dim=64, num_classes=NUM_CLASSES, in_channels=IN_CHANNELS).to(device)
x = torch.randn(2, IN_CHANNELS, IMG_SIZE, IMG_SIZE, device=device)
t = torch.tensor([999, 7], device=device)       # two different timesteps
labels = torch.tensor([3, 8], device=device)    # two different classes

print("input shape: ", tuple(x.shape))
print("output shape:", tuple(m(x, t, labels).shape))
print(f"{sum(p.numel() for p in m.parameters()):,} parameters (at base_ch=8)")

del m, x, t, labels   # free the memory again

### 4.3 🎯-🏋️ Train the UNet

Now that we have defined our denoising autoencoder, we can train it. For each batch we sample a random timestep $t$, corrupt the images with the corresponding noise level, and ask the model to predict that noise. The loss is simply the MSE between predicted and actual noise.

In [ ]:

# --- Initialize the model which we use to predict the noise.
denoising_model = SimpleUNet(base_ch=64, emb_dim=64, num_classes=NUM_CLASSES, in_channels=IN_CHANNELS).to(device)
denoising_model.train()
optimizer = optim.AdamW(denoising_model.parameters(), lr=2e-4)

# --- Training loop, learning to reverse the noise. ---
epochs = 30

for epoch in range(1, epochs + 1):
    for x_0, labels in train_loader:
        optimizer.zero_grad(set_to_none=True)
        x_0 = x_0.to(device)
        labels = labels.to(device)
        batch_dimension = x_0.size(0)
        t = torch.randint(0, TIMESTEPS, (batch_dimension,), device=device).long() # choose random timestep for every sample from the input

        # 🎯 Add noise using the forward process (Hint: you already implemented this in exercise 3.1)
        x_t, noise = ...

        # 🎯 Predict the added noise, conditioned on the class label
        # Hint: pass the noisy image, timesteps, and labels into the denoising model
        predicted_noise = ...

        # 🎯Compute the (mean squared error) loss between the predicted_noise and the real noise
        # Hint: use F.mse_loss to compute the MSE loss
        loss = ...

        # 🎯 Backward pass and optimization step
        ...
        ...

    print(f"Finished epoch {epoch}. Current loss: loss={loss.item():.4f}")

print("Training is finished!")

### 4.4 💾 Save and Load the Model

Training takes a while, and a Colab runtime can disconnect at any time. `torch.save` writes the learned weights to disk so you do not have to retrain. `torch.load` reads them back into a freshly built `SimpleUNet`.

Only the weights are saved, not the architecture, so the model you load into must be built with the same constructor arguments as the one you trained. Download the `.pt` file from the Colab file browser if you want to keep it beyond this session.

In [ ]:
# Save the trained weights so you do not lose them if the runtime disconnects.
CHECKPOINT_PATH = "ddpm_fashion.pt"
torch.save(denoising_model.state_dict(), CHECKPOINT_PATH)
print(f"Saved weights to {CHECKPOINT_PATH}")

In [ ]:
# Rebuild the architecture with the SAME arguments, then load the weights back in.
denoising_model = SimpleUNet(
    base_ch=64, emb_dim=64, num_classes=NUM_CLASSES, in_channels=IN_CHANNELS
).to(device)
denoising_model.load_state_dict(torch.load(CHECKPOINT_PATH, map_location=device))
denoising_model.eval()
print(f"Loaded weights from {CHECKPOINT_PATH}")

---
## 5 🎨 Generate New Fashion

Training is done. Now comes the fun part. We sample new images by starting from **pure Gaussian noise** and running the reverse diffusion process around a 1000 steps, guided by the trained UNet.

Because the model is class-conditional, we can ask it to generate any of the 10 clothing categories. Change `class_label` to any number from 0 to 9 and watch the model invent a new garment from static.

### 5.1 🔁 The Sampling Equations

Training taught the model to predict the sampled noise. Now we use that prediction to iteratively denoise, stepping **backwards** from $t = T-1$ (pure noise) to $t = 0$ (clean image).

**The key idea:** at each step $t$ we ask the trained UNet *"what noise pattern do you think was sampled to produce this image?"*. That prediction is $\varepsilon_\theta(x_t, t, y)$. We then use it to compute $x_{t-1}$, and repeat 1,000 times.

**Step 1: estimate the posterior mean.** This is the sampling equation. It is the fixed formula that turns the UNet's noise prediction into one step backwards:

$$\mu_\theta(x_t, t, y) = \frac{1}{\sqrt{\alpha_t}} \left( x_t - \frac{1-\alpha_t}{\sqrt{1 - \bar{\alpha}_{t}}} \underbrace{\varepsilon_\theta(x_t, t, y)}_{\text{UNet output}} \right)$$

$\mu_\theta$ is the mean of the distribution over $x_{t-1}$: our best guess for the image with a little less noise, one small step closer to the finished garment. Nothing in this formula is learned. The only piece the UNet supplies is $\varepsilon_\theta$.

**Step 2: add a pinch of randomness.** If we always took the exact mean, every generation run would produce the same image. To keep outputs diverse, we add a small random kick:

$$x_{t-1} = \mu_\theta(x_t, t, y) + \underbrace{\sigma_t \cdot z}_{\text{diversity noise}}, \qquad z \sim \mathcal{N}(0, I)$$

where $z$ is fresh random noise and $\sigma_t = \sqrt{\tilde{\beta}_t}$ scales it down. At the very last step ($t = 0$) we skip this term entirely, so the final image is not grainy.

> **In plain English:** start from static. At each step, the UNet looks at the current blurry mess and predicts the noise pattern inside it. The sampling equation removes most of that noise, we add a tiny bit of fresh randomness for variety, and repeat until a recognizable garment emerges.

In [ ]:
# Precompute helpful buffers for sampling
alphas_bar_previous = torch.cat([torch.tensor([1.0], device=device), alphas_bar[:-1]], dim=0)
sqrt_alphas_inverse = torch.sqrt(1.0 / alphas)

# Posterior mean coefficient: (1 - alpha_t) / sqrt(1 - alpha_bar_t)
coef_eps = (1 - alphas) / sqrt_one_minus_alphas_bar

# Posterior variance
posterior_variance = (1 - alphas) * (1 - alphas_bar_previous) / (1 - alphas_bar)

### 🎯 5.2 Implement the denoising step

Complete the `mean` computation inside `sample`. This is the sampling equation from 5.1: the model's best estimate of a slightly cleaner image at each reverse step.

> 💡 Hints:
> - The formula is: $\mu_\theta(x_t, t, y) = \dfrac{1}{\sqrt{\alpha_t}} \left( x_t - \dfrac{1-\alpha_t}{\sqrt{1 - \bar{\alpha}_{t}}} \cdot \varepsilon_\theta \right)$
> - `coef_eps[timestep]` gives you $\dfrac{1-\alpha_t}{\sqrt{1-\bar{\alpha}_t}}$, and `sqrt_alphas_inverse[timestep]` gives you $\dfrac{1}{\sqrt{\alpha_t}}$
> - `denoising_model(x_t, t, labels)` is $\varepsilon_\theta$, the noise **predicted** by the UNet. It is not the same thing as the true $\varepsilon$ from Section 2. The gap between them is exactly what the training loss minimizes.

In [ ]:
@torch.no_grad()
def sample(denoising_model, num_samples=16, class_label=0, record_timesteps=None):
    """
    Reverse diffusion: start from pure noise and step back to a clean image.

    Args:
        denoising_model:  the trained UNet, which predicts the sampled noise.
        num_samples:      how many images to generate in parallel.
        class_label:      which clothing class to condition on (same for all samples).
        record_timesteps: list of timesteps at which to snapshot x_t, or None for no trace.

    Returns:
        x_0:   (num_samples, IN_CHANNELS, IMG_SIZE, IMG_SIZE), the generated images.
        trace: list of (t, x_t) snapshots in descending t order, empty if
               record_timesteps is None. The snapshot tagged t IS x_t, that is,
               the state BEFORE the model's step at that timestep.
    """
    denoising_model.eval()

    # Start from random noise
    x_t = torch.randn(num_samples, IN_CHANNELS, IMG_SIZE, IMG_SIZE, device=device)

    # Class label tensor, same label for all samples in this batch
    labels = torch.full((num_samples,), class_label, device=device, dtype=torch.long)

    record = set(record_timesteps) if record_timesteps is not None else set()
    trace = []

    # Now we reverse the noising
    for timestep in reversed(range(TIMESTEPS)):
        if timestep in record:
            trace.append((timestep, x_t.detach().clone()))

        t = torch.full((num_samples,), timestep, device=device, dtype=torch.long)

        # 🎯 Denoise the image
        # Hint: use the formula and the hints above
        # Hint: epsilon (the predicted noise) is given by the output of the denoising model
        epsilon = ...
        mean = ...

        if timestep > 0:
            # Add noise for sample diversity
            noise = torch.randn_like(x_t)
            var = posterior_variance[timestep]
            
            # 🎯 Compute x_{t-1} using the mean, var and noise
            # Hint: recall that we need to torch.sqrt() the variance when adding noise
            x_t = ...
        else:
            # 🎯 Compute final x_t, here we don't add noise
            x_t = ...

    return x_t, trace

In [ ]:
#@title Display Samples { display-mode: "form" }
for class_label in range(NUM_CLASSES):
    samples, _ = sample(denoising_model, num_samples=4, class_label=class_label)
    utils.show_images(samples.cpu(), titles=[CLASS_NAMES[class_label]] * 4)

---
## 6 🎞️ Watch It Denoise

A single image, visualized at 10 checkpoints across the full denoising journey, from pure noise all the way to the finished garment.

**What to look for:**
- **High $t$**: indistinguishable static; the model is just getting started
- **Mid $t$**: rough shapes and outlines begin to emerge
- **Low $t$**: fine texture and detail snap into place in the final steps

This is the reverse of Section 3: the same 10-step window, but running backwards.

In [ ]:
#@title Display Timestep Traces { display-mode: "form" }
vis_timesteps = torch.linspace(TIMESTEPS - 1, 0, 10).long().tolist()

for class_label in range(NUM_CLASSES):
    _, trace = sample(
        denoising_model, num_samples=1, class_label=class_label, record_timesteps=vis_timesteps
    )
    trace_dict = {t: x_t for t, x_t in trace}
    imgs = [trace_dict[t].squeeze(0).cpu() for t in vis_timesteps]
    titles = [f"t={t}" for t in vis_timesteps]
    print(f"Class {class_label}: {CLASS_NAMES[class_label]}")
    utils.show_images(torch.stack(imgs), titles=titles)

---
## 7 🌐 Interactive Web App

Run the cell below to launch the full fashion magazine generator. It uses the trained DDPM model to generate images and an LLM via OpenRouter for the article text, identical to the standalone fashion app.

Enter your OpenRouter API key (`sk-or-...`) in the **OpenRouter API Key** field at the top of the app sidebar and click **Save Key**. The key stays hidden while you type. Alternatively, store it as a Colab secret named `OPENROUTER_API_KEY`.

In [ ]:
#@title Load your OpenRouter API key (optional) { display-mode: "form" }
# You can type your OpenRouter key ("sk-or-...") directly in the app below; it stays hidden while you type.
# Optional: to skip typing it every time, add it as a Colab secret named OPENROUTER_API_KEY (🔑 panel on the left).
from google.colab import userdata

try:
    API_KEY = userdata.get("OPENROUTER_API_KEY")
    print("✅ OpenRouter key loaded from Colab secrets.")
except Exception:
    API_KEY = None
    print("No OPENROUTER_API_KEY secret found; enter your key in the app instead.")

In [ ]:
#@title Run the Web App { display-mode: "form" }
REPO    = "eth-bmai-hs26/w3-project-public"
BRANCH  = "main/notebook"


# ── 1. Download app_colab.py and dataset from GitHub ──────────────────────────

BASE_RAW = f"https://raw.githubusercontent.com/{REPO}/refs/heads/{BRANCH}"

subprocess.run(["curl", "-fsSL", "-O", f"{BASE_RAW}/visualization/app_colab.py"], check=True)

os.makedirs("/content/Data", exist_ok=True)
subprocess.run(["curl", "-fsSL", "-o", "/content/Data/fashion_data_2018_2022.xls",
                f"{BASE_RAW}/visualization/Data/fashion_data_2018_2022.xls"], check=True)

import importlib, app_colab

importlib.reload(app_colab)
print("app_colab.py loaded.")

# ── 2. Image generation: uses denoising_model trained in section 4 ──────────
def _generate_ddpm_image(class_label=0):
    generated, _ = sample(denoising_model, num_samples=1, class_label=int(class_label) % NUM_CLASSES)
    generated = (generated.clamp(-1, 1) + 1) / 2.0
    return app_colab.tensor_to_b64(generated[0])

# ── 3. Launch: displays the app directly in this cell output ─────────────────
app_colab.register_all(
    generate_fn=_generate_ddpm_image,
    api_key=API_KEY,
    branch=BRANCH,
    repo=REPO,
)

---
## 8 💬 Discussion

**The goal of this notebook was to teach you how to build a generative model from scratch.** You now understand how diffusion works end-to-end: how to corrupt data with noise, how to train a neural network to reverse that process, and how to sample new images from pure static.

Diffusion models are a novel and elegant way to generate data from complex, unknown distributions: the model never sees the true data distribution directly, yet learns to reproduce it by simply practicing to reverse noise.

This raises a practical question every ML engineer faces: **build vs. use?**

- **Build your own** when you need full control, have domain-specific data, or want to deeply understand the system, as you just did.
- **Rely on a 3rd-party model** (via API) when quality, speed, and cost matter more than ownership, and when a strong pre-trained model already exists for your task.

In our fashion app, we *could* plug in the toy DDPM you just trained, but a production-grade API will produce far better images with far less effort. Knowing when to make that trade-off is a core skill in applied ML.

> 🤔 **Reflection question:** How could we handle high-dimensional images with diffusion models?
>
> 📄 Paper: [High-Resolution Image Synthesis with Latent Diffusion Models (Rombach et al., 2022)](https://arxiv.org/abs/2112.10752)